<a href="https://colab.research.google.com/github/bhumikashiriya-dot/Data-Engineering/blob/main/practical_2_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import sqlite3
import pandas as pd

# Create a connection to an in-memory SQLite database
conn = sqlite3.connect(':memory:')
cursor = conn.cursor()

# Create table Student_Data
cursor.execute('''
CREATE TABLE Student_Data
(
    Student_ID INT,
    Student_Name VARCHAR(50),
    Age INT,
    City VARCHAR(50),
    Marks DECIMAL(5,2)
);
''')

# Insert records into Student_Data
cursor.executemany('''
INSERT INTO Student_Data (Student_ID, Student_Name, Age, City, Marks)
VALUES (?, ?, ?, ?, ?);
''', [
    (1, 'Rahul', 20, 'Mumbai', 85),
    (2, 'Sneha', 21, 'mumbai', 92),
    (3, 'Amit', None, 'Pune', 78),
    (4, 'Priya', 20, None, 88),
    (5, 'Rohan', 22, 'PUNE', 65)
])
conn.commit()

# 1. Extract Step
df_extract = pd.read_sql_query('''
SELECT
    Student_ID,
    Student_Name,
    Age,
    City,
    Marks
FROM Student_Data;
''', conn)
print("--- EXTRACTED DATA ---")
print(df_extract)

# 2. Transform Step
df_transform = pd.read_sql_query('''
SELECT
    Student_ID,
    Student_Name,
    COALESCE(Age, 0) AS Age,
    CASE
        WHEN UPPER(City) = 'MUMBAI' THEN 'Mumbai'
        WHEN UPPER(City) = 'PUNE' THEN 'Pune'
        ELSE COALESCE(City, 'Unknown')
    END AS City,
    Marks,
    CASE
        WHEN Marks >= 85 THEN 'Excellent'
        WHEN Marks >= 70 THEN 'Good'
        ELSE 'Needs Improvement'
    END AS Performance_Category
FROM Student_Data;
''', conn)
print("\n--- TRANSFORMED DATA ---")
print(df_transform)

# 3. Load Step
cursor.execute('''
CREATE TABLE Student_Final
(
    Student_ID INT,
    Student_Name VARCHAR(50),
    Age INT,
    City VARCHAR(50),
    Marks DECIMAL(5,2),
    Performance_Category VARCHAR(30)
);
''')

cursor.execute('''
INSERT INTO Student_Final
(
    Student_ID,
    Student_Name,
    Age,
    City,
    Marks,
    Performance_Category
)
SELECT
    Student_ID,
    Student_Name,
    COALESCE(Age, 0),
    CASE
        WHEN UPPER(City) = 'MUMBAI' THEN 'Mumbai'
        WHEN UPPER(City) = 'PUNE' THEN 'Pune'
        ELSE COALESCE(City, 'Unknown')
    END,
    Marks,
    CASE
        WHEN Marks >= 85 THEN 'Excellent'
        WHEN Marks >= 70 THEN 'Good'
        ELSE 'Needs Improvement'
    END
FROM Student_Data;
''')
conn.commit()

df_final = pd.read_sql_query('SELECT * FROM Student_Final;', conn)
print("\n--- LOADED DATA (Student_Final) ---")
print(df_final)